In [16]:
!pip install trimesh
!pip install thingi10k

INFO: pip is looking at multiple versions of thingi10k to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.6 MB/s  0:00:00m eta 0:00:01
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 47.4 MB/s  0:00:00
Using c

In [21]:
import os
import urllib.request
from urllib.error import HTTPError

output_folder = "My_Thesis_Dataset"
os.makedirs(output_folder, exist_ok=True)

print("Starting direct download of 30 CAD models from Thingi10K...")

# The official Hugging Face mirror URL structure
file_id = 100000
downloaded_count = 0
target_count = 30

while downloaded_count < target_count:
    filename = f"{file_id}.stl"
    url = f"https://huggingface.co/datasets/Thingi10K/Thingi10K/resolve/main/raw_meshes/{filename}"
    destination = os.path.join(output_folder, filename)
    
    try:
        # Download the file directly
        urllib.request.urlretrieve(url, destination)
        
        # Check if the file is valid (larger than 1KB to avoid empty placeholder files)
        if os.path.getsize(destination) > 1000: 
            print(f"[{downloaded_count + 1}/{target_count}] Successfully downloaded {filename}")
            downloaded_count += 1
        else:
            os.remove(destination) 
            
    except HTTPError:
        # File doesn't exist at this specific ID on the server, skip to the next one
        pass
    except Exception as e:
        print(f"Network error: {e}")
        break
        
    file_id += 1

print(f"\nSuccess! Your folder '{output_folder}' now has {downloaded_count} CAD models ready for training.")

Starting direct download of 30 CAD models from Thingi10K...
[1/30] Successfully downloaded 100026.stl
[2/30] Successfully downloaded 100027.stl
[3/30] Successfully downloaded 100028.stl
[4/30] Successfully downloaded 100029.stl
[5/30] Successfully downloaded 100030.stl
[6/30] Successfully downloaded 100031.stl
[7/30] Successfully downloaded 100032.stl
[8/30] Successfully downloaded 100033.stl
[9/30] Successfully downloaded 100034.stl
[10/30] Successfully downloaded 100035.stl
[11/30] Successfully downloaded 100036.stl
[12/30] Successfully downloaded 100037.stl
[13/30] Successfully downloaded 100040.stl
[14/30] Successfully downloaded 100045.stl
[15/30] Successfully downloaded 100046.stl
[16/30] Successfully downloaded 100047.stl
[17/30] Successfully downloaded 100070.stl
[18/30] Successfully downloaded 100071.stl
[19/30] Successfully downloaded 100072.stl
[20/30] Successfully downloaded 100073.stl
[21/30] Successfully downloaded 100075.stl
[22/30] Successfully downloaded 100077.stl
[23

In [22]:
!ls dataset

100026.stl         airplane_0009.off  bookshelf_0008.off chair_0006.off
100027.stl         airplane_0010.off  bookshelf_0009.off chair_0007.off
100028.stl         bathtub_0001.off   bookshelf_0010.off chair_0008.off
100029.stl         bathtub_0002.off   bottle_0001.off    chair_0009.off
100030.stl         bathtub_0003.off   bottle_0002.off    chair_0010.off
100031.stl         bathtub_0004.off   bottle_0003.off    cone_0001.off
100032.stl         bathtub_0005.off   bottle_0004.off    cone_0002.off
100033.stl         bathtub_0006.off   bottle_0005.off    cone_0003.off
100034.stl         bathtub_0007.off   bottle_0006.off    cone_0004.off
100035.stl         bathtub_0008.off   bottle_0007.off    cone_0005.off
100036.stl         bathtub_0009.off   bottle_0008.off    cone_0006.off
100037.stl         bathtub_0010.off   bottle_0009.off    cone_0007.off
100040.stl         bed_0001.off       bottle_0010.off    cone_0008.off
100045.stl         bed_0002.off       bowl_0001.off      cone_0009.off
1

In [ ]:
import os
import torch
import trimesh
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data

class CustomMeshDataset(Dataset):
    """
    Loads any mix of .obj, .ply, and .stl files from a local folder.
    Normalizes them, calculates geometric normals, and samples 1024 points.
    """
    def __init__(self, folder_path, num_points=1024, k_neighbors=20):
        self.folder_path = folder_path
        self.num_points = num_points
        self.k = k_neighbors
        
        # Scan the folder for all 3D files
        self.file_list = [
            os.path.join(folder_path, f) for f in os.listdir(folder_path) 
            if f.lower().endswith(('.obj', '.ply', '.stl'))
        ]
        print(f"Dataset Initialized: Found {len(self.file_list)} 3D models in '{folder_path}'")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filepath = self.file_list[idx]
        
        # 1. Load the mesh
        mesh = trimesh.load(filepath, process=False)
        pos = torch.tensor(mesh.vertices, dtype=torch.float32)
        orig_faces = torch.tensor(mesh.faces, dtype=torch.long)
        
        # 2. Center and Normalize (Crucial for generalizing across different shapes)
        pos = pos - pos.mean(dim=0)
        max_dist = torch.max(torch.sqrt(torch.sum(pos**2, dim=1)))
        if max_dist > 0:
            pos = pos / max_dist
            
        # 3. Compute Normals Natively
        v0, v1, v2 = pos[orig_faces[:,0]], pos[orig_faces[:,1]], pos[orig_faces[:,2]]
        face_n = torch.cross(v1 - v0, v2 - v0, dim=1)
        vert_n = torch.zeros_like(pos)
        vert_n.scatter_add_(0, orig_faces[:,0:1].expand(-1,3), face_n)
        vert_n.scatter_add_(0, orig_faces[:,1:2].expand(-1,3), face_n)
        vert_n.scatter_add_(0, orig_faces[:,2:3].expand(-1,3), face_n)
        norm = F.normalize(vert_n, p=2, dim=1)
        
        # 4. Sample exactly 1024 points to keep the GNN brain happy
        num_vertices = pos.shape[0]
        if num_vertices >= self.num_points:
            # Downsample randomly
            sample_idx = torch.randperm(num_vertices)[:self.num_points]
        else:
            # Upsample with replacement if a mesh has fewer than 1024 points
            sample_idx = torch.randint(0, num_vertices, (self.num_points,))
            
        pos_sampled = pos[sample_idx]
        norm_sampled = norm[sample_idx]
        
        # 5. Build the KNN Edge Index native to M4
        dist = torch.cdist(pos_sampled, pos_sampled)
        _, knn_idx = torch.topk(dist, k=self.k + 1, largest=False)
        edges = torch.stack([
            torch.arange(self.num_points).view(-1, 1).repeat(1, self.k).flatten(), 
            knn_idx[:, 1:].flatten()
        ], dim=0)
        
        # Return a PyTorch Geometric Data object
        return Data(pos=pos_sampled, normal=norm_sampled, edge_index=edges)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data
from torch.utils.data import Dataset
import os
import trimesh

# --- 1. SETTINGS & HARDWARE ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MODEL_PATH = "universal_vsa_brain.pt"
DATASET_FOLDER = "dataset"
NUM_PROXIES = 800  
print(f"Running High-Res Generalized Training on: {device}")

# --- 2. CUSTOM DATASET LOADER ---
class CustomMeshDataset(Dataset):
    # INCREASED: Now sampling 4096 points for higher resolution learning
    def __init__(self, folder_path, num_points=4096, k_neighbors=20):
        self.num_points = num_points
        self.k = k_neighbors
        self.file_list = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                          if f.lower().endswith(('.obj', '.ply', '.stl', '.off'))]
        print(f"Dataset Initialized: Found {len(self.file_list)} models.")

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        mesh = trimesh.load(self.file_list[idx], process=False)
        pos = torch.tensor(mesh.vertices, dtype=torch.float32)
        orig_faces = torch.tensor(mesh.faces, dtype=torch.long)
        
        # Center & Normalize
        pos = pos - pos.mean(dim=0)
        max_dist = torch.max(torch.sqrt(torch.sum(pos**2, dim=1)))
        if max_dist > 0: pos = pos / max_dist
            
        # Compute Normals Native PyTorch
        v0, v1, v2 = pos[orig_faces[:,0]], pos[orig_faces[:,1]], pos[orig_faces[:,2]]
        face_n = torch.cross(v1 - v0, v2 - v0, dim=1)
        vert_n = torch.zeros_like(pos)
        vert_n.scatter_add_(0, orig_faces[:,0:1].expand(-1,3), face_n)
        vert_n.scatter_add_(0, orig_faces[:,1:2].expand(-1,3), face_n)
        vert_n.scatter_add_(0, orig_faces[:,2:3].expand(-1,3), face_n)
        norm = F.normalize(vert_n, p=2, dim=1)
        
        # Sample exactly 4096 points
        num_vertices = pos.shape[0]
        if num_vertices >= self.num_points:
            sample_idx = torch.randperm(num_vertices)[:self.num_points]
        else:
            sample_idx = torch.randint(0, num_vertices, (self.num_points,))
            
        pos_sampled, norm_sampled = pos[sample_idx], norm[sample_idx]
        
        # KNN Edge Index
        dist = torch.cdist(pos_sampled, pos_sampled)
        _, knn_idx = torch.topk(dist, k=self.k + 1, largest=False)
        edges = torch.stack([torch.arange(self.num_points).view(-1, 1).repeat(1, self.k).flatten(), 
                             knn_idx[:, 1:].flatten()], dim=0)
        
        return Data(pos=pos_sampled, normal=norm_sampled, edge_index=edges)

# --- 3. ARCHITECTURE & MATH ---
class NeuralVSALayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(NeuralVSALayer, self).__init__(aggr='add') 
        self.mlp = nn.Sequential(nn.Linear(in_channels * 2, 64), nn.ReLU(), nn.Linear(64, out_channels))
    def forward(self, x, edge_index): return self.propagate(edge_index, x=x)
    def message(self, x_i, x_j): return self.mlp(torch.cat([x_i, x_j], dim=1))

class SoftAssignmentNetwork(nn.Module):
    def __init__(self, hidden_channels, num_proxies):
        super().__init__()
        self.predictor = nn.Sequential(nn.Linear(hidden_channels, 64), nn.ReLU(), nn.Linear(64, num_proxies))
    # Added the tau (temperature) parameter here!
    def forward(self, x, tau=1.0): 
        return F.softmax(self.predictor(x) / tau, dim=1)

def compute_quadrics(points, normals):
    d = -torch.sum(points * normals, dim=1, keepdim=True)
    v = torch.cat([normals, d], dim=1)
    return torch.bmm(v.unsqueeze(2), v.unsqueeze(1)) 

def differentiable_qem_loss(points, quadrics, S):
    N, K = S.shape
    weights_sum = S.sum(dim=0).unsqueeze(1) + 1e-8 
    proxies = torch.matmul(S.T, points) / weights_sum 
    proxies_h = torch.cat([proxies, torch.ones(K, 1, device=points.device)], dim=1)
    proxies_expanded = proxies_h.unsqueeze(0).expand(N, K, 4)
    proxy_Q_product = torch.einsum('nkd,ndw->nkw', proxies_expanded, quadrics)
    errors = torch.einsum('nkw,nkw->nk', proxy_Q_product, proxies_expanded)
    return torch.sum(errors * S)

gnn = NeuralVSALayer(in_channels=6, out_channels=64).to(device)
assigner = SoftAssignmentNetwork(hidden_channels=64, num_proxies=NUM_PROXIES).to(device)

# --- 4. TRAINING WITH EARLY STOPPING ---
if not os.path.exists(MODEL_PATH):
    print("\nStarting Training Phase...")
    dataset = CustomMeshDataset(DATASET_FOLDER)
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    
    # LOWERED: LR from 0.005 to 0.001 for stable proxy distribution
    optimizer = optim.Adam(list(gnn.parameters()) + list(assigner.parameters()), lr=0.001)
    
    best_loss = float('inf')
    patience_counter = 0
    patience = 50 # INCREASED patience
    
    for epoch in range(500):
        total_loss = 0
        for data in loader:
            data = data.to(device)
            optimizer.zero_grad()
            Q = compute_quadrics(data.pos, data.normal)
            S = assigner(gnn(torch.cat([data.pos, data.normal], dim=1), data.edge_index))
            loss = differentiable_qem_loss(data.pos, Q, S)
            loss.backward(); optimizer.step(); total_loss += loss.item()
            
        avg_loss = total_loss / len(dataset)
        print(f"Epoch {epoch+1:03d} | Avg Loss: {avg_loss:.4f}", end="")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save({'gnn': gnn.state_dict(), 'assigner': assigner.state_dict()}, MODEL_PATH)
            print("  <- New Best! Saved.")
        else:
            patience_counter += 1
            print(f"  (Patience: {patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered!")
                break
                
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    gnn.load_state_dict(checkpoint['gnn']); assigner.load_state_dict(checkpoint['assigner'])
else:
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    gnn.load_state_dict(checkpoint['gnn']); assigner.load_state_dict(checkpoint['assigner'])
    print(f"\nLoaded pre-trained generalized weights from {MODEL_PATH}")

# # --- 5. HYBRID INFERENCE (Annealed TTO) ---
# def extract_hybrid_mesh(input_mesh_path):
#     print(f"\n--- Hybrid Inference Phase: {input_mesh_path} ---")
#     mesh = trimesh.load(input_mesh_path, process=False)
#     if len(mesh.faces) == 0: raise ValueError("No faces found!")
        
#     pos_dense = torch.tensor(mesh.vertices, dtype=torch.float32).to(device)
#     orig_faces = torch.tensor(mesh.faces, dtype=torch.long).to(device)
    
#     # Normalize
#     pos_dense = pos_dense - pos_dense.mean(dim=0)
#     pos_dense = pos_dense / torch.max(torch.sqrt(torch.sum(pos_dense**2, dim=1)))
    
#     # Dense Normals & Quadrics
#     v0, v1, v2 = pos_dense[orig_faces[:,0]], pos_dense[orig_faces[:,1]], pos_dense[orig_faces[:,2]]
#     face_n = torch.cross(v1 - v0, v2 - v0, dim=1)
#     vert_n_dense = torch.zeros_like(pos_dense)
#     for i in range(3): vert_n_dense.scatter_add_(0, orig_faces[:,i:i+1].expand(-1,3), face_n)
#     norm_dense = F.normalize(vert_n_dense, p=2, dim=1)
    
#     d = -torch.sum(pos_dense * norm_dense, dim=1, keepdim=True)
#     v = torch.cat([norm_dense, d], dim=1)
#     Q_dense = torch.bmm(v.unsqueeze(2), v.unsqueeze(1))
    
#     # Sample 4096 points
#     sample_count = min(4096, pos_dense.shape[0])
#     sample_idx = torch.randperm(pos_dense.shape[0])[:sample_count]
#     pos_s, norm_s, Q_s = pos_dense[sample_idx], norm_dense[sample_idx], Q_dense[sample_idx]
    
#     # INCREASED: k=31 to give the network a wider view during fine-tuning
#     dist_s = torch.cdist(pos_s, pos_s)
#     _, knn_idx = torch.topk(dist_s, k=31, largest=False)
#     edges = torch.stack([torch.arange(len(sample_idx), device=device).view(-1,1).repeat(1,30).flatten(), knn_idx[:,1:].flatten()], dim=0)

#     # ==========================================
#     # TEST-TIME OPTIMIZATION (With Annealing)
#     # ==========================================
#     print("Fine-tuning with Temperature Annealing (TTO)...")
#     gnn.train(); assigner.train()
    
#     # INCREASED: LR to 0.005 to encourage more proxy movement
#     tto_optimizer = optim.Adam(list(gnn.parameters()) + list(assigner.parameters()), lr=0.005)
#     features_s = torch.cat([pos_s, norm_s], dim=1)
    
#     num_tto_steps = 500
#     tau_start = 1.0
#     tau_end = 0.1
    
#     for i in range(num_tto_steps): 
#         tto_optimizer.zero_grad()
        
#         # Calculate current temperature (Linear decay)
#         current_tau = tau_start - (tau_start - tau_end) * (i / num_tto_steps)
        
#         S_sampled = assigner(gnn(features_s, edges), tau=current_tau)
#         loss = differentiable_qem_loss(pos_s, Q_s, S_sampled)
#         loss.backward()
#         tto_optimizer.step()
        
#         if (i+1) % 50 == 0: 
#             print(f"  TTO Step {i+1} | Tau: {current_tau:.3f} | Loss: {loss.item():.4f}")

#     # ==========================================
#     # FINAL EXTRACTION
#     # ==========================================
#     gnn.eval(); assigner.eval()
#     with torch.no_grad():
#         print("Extracting final annealed mesh...")
#         # Use the final, coldest temperature for the hard prediction
#         S_sampled = assigner(gnn(features_s, edges), tau=tau_end)
#         proxies = torch.matmul(S_sampled.T, pos_s) / (S_sampled.sum(dim=0).unsqueeze(1) + 1e-8)
        
#         dist_to_proxies = torch.cdist(pos_dense, proxies)
#         vertex_to_proxy = torch.argmin(dist_to_proxies, dim=1)
        
#         sim_f = vertex_to_proxy[orig_faces]
#         mask = (sim_f[:,0] != sim_f[:,1]) & (sim_f[:,1] != sim_f[:,2]) & (sim_f[:,0] != sim_f[:,2])
#         final_f = torch.unique(torch.sort(sim_f[mask], dim=1)[0], dim=0)

#     output_filename = f"hybrid_output_{NUM_PROXIES}.obj"
#     with open(output_filename, "w") as f:
#         for v in proxies.cpu().numpy(): f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
#         for face in (final_f.cpu().numpy() + 1): f.write(f"f {face[0]} {face[1]} {face[2]}\n")
#     print(f"Success! {final_f.shape[0]} faces extracted and saved to {output_filename}")

# # Run Inference
# bunny_path = "dataset/bunny.ply"
# if os.path.exists(bunny_path):
#     extract_hybrid_mesh(bunny_path)

Running High-Res Generalized Training on: mps

Starting Training Phase...
Dataset Initialized: Found 155 models.
Epoch 001 | Avg Loss: 200.5239  <- New Best! Saved.
Epoch 002 | Avg Loss: 124.3178  <- New Best! Saved.
Epoch 003 | Avg Loss: 116.6994  <- New Best! Saved.
Epoch 004 | Avg Loss: 114.5853  <- New Best! Saved.
Epoch 005 | Avg Loss: 111.9134  <- New Best! Saved.
Epoch 006 | Avg Loss: 111.6448  <- New Best! Saved.
Epoch 007 | Avg Loss: 110.4877  <- New Best! Saved.
Epoch 008 | Avg Loss: 110.0830  <- New Best! Saved.
Epoch 009 | Avg Loss: 109.7919  <- New Best! Saved.
Epoch 010 | Avg Loss: 108.6706  <- New Best! Saved.
Epoch 011 | Avg Loss: 108.6650  <- New Best! Saved.
Epoch 012 | Avg Loss: 106.2481  <- New Best! Saved.
Epoch 013 | Avg Loss: 106.2628  (Patience: 1/50)
Epoch 014 | Avg Loss: 107.5276  (Patience: 2/50)
Epoch 015 | Avg Loss: 106.6202  (Patience: 3/50)
Epoch 016 | Avg Loss: 106.7634  (Patience: 4/50)
Epoch 017 | Avg Loss: 104.9783  <- New Best! Saved.
Epoch 018 | Avg

KeyboardInterrupt: 

***Supervised Inference***

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import MessagePassing
import os
import trimesh

# --- 1. SETTINGS & HARDWARE ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MODEL_PATH = "universal_vsa_brain.pt"

# THE MAGIC BULLET: Over-provisioning to 800 to yield ~300 faces
NUM_PROXIES = 800  
print(f"Running Stable Over-Provisioned Inference on: {device}")

# --- 2. STABLE ARCHITECTURE ---
class NeuralVSALayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(NeuralVSALayer, self).__init__(aggr='add') 
        self.mlp = nn.Sequential(nn.Linear(in_channels * 2, 64), nn.ReLU(), nn.Linear(64, out_channels))
    def forward(self, x, edge_index): return self.propagate(edge_index, x=x)
    def message(self, x_i, x_j): return self.mlp(torch.cat([x_i, x_j], dim=1))

class SoftAssignmentNetwork(nn.Module):
    def __init__(self, hidden_channels, num_proxies):
        super().__init__()
        self.predictor = nn.Sequential(nn.Linear(hidden_channels, 64), nn.ReLU(), nn.Linear(64, num_proxies))
    
    # Using the natively stable PyTorch Softmax
    def forward(self, x, tau=1.0): 
        return F.softmax(self.predictor(x) / tau, dim=1)

def compute_quadrics(points, normals):
    d = -torch.sum(points * normals, dim=1, keepdim=True)
    v = torch.cat([normals, d], dim=1)
    return torch.bmm(v.unsqueeze(2), v.unsqueeze(1)) 

def differentiable_qem_loss(points, quadrics, S):
    N, K = S.shape
    weights_sum = S.sum(dim=0).unsqueeze(1) + 1e-8 
    proxies = torch.matmul(S.T, points) / weights_sum 
    proxies_h = torch.cat([proxies, torch.ones(K, 1, device=points.device)], dim=1)
    proxies_expanded = proxies_h.unsqueeze(0).expand(N, K, 4)
    proxy_Q_product = torch.einsum('nkd,ndw->nkw', proxies_expanded, quadrics)
    errors = torch.einsum('nkw,nkw->nk', proxy_Q_product, proxies_expanded)
    return torch.sum(errors * S)

gnn = NeuralVSALayer(in_channels=6, out_channels=64).to(device)
assigner = SoftAssignmentNetwork(hidden_channels=64, num_proxies=NUM_PROXIES).to(device)

# --- 3. HYBRID INFERENCE ---
def extract_hybrid_mesh(input_mesh_path):
    print(f"\n--- Hybrid Inference Phase: {input_mesh_path} ---")
    mesh = trimesh.load(input_mesh_path, process=False)
    if len(mesh.faces) == 0: raise ValueError("No faces found!")
        
    pos_dense = torch.tensor(mesh.vertices, dtype=torch.float32).to(device)
    orig_faces = torch.tensor(mesh.faces, dtype=torch.long).to(device)
    
    pos_dense = pos_dense - pos_dense.mean(dim=0)
    pos_dense = pos_dense / torch.max(torch.sqrt(torch.sum(pos_dense**2, dim=1)))
    
    v0, v1, v2 = pos_dense[orig_faces[:,0]], pos_dense[orig_faces[:,1]], pos_dense[orig_faces[:,2]]
    face_n = torch.cross(v1 - v0, v2 - v0, dim=1)
    vert_n_dense = torch.zeros_like(pos_dense)
    for i in range(3): vert_n_dense.scatter_add_(0, orig_faces[:,i:i+1].expand(-1,3), face_n)
    norm_dense = F.normalize(vert_n_dense, p=2, dim=1)
    
    d = -torch.sum(pos_dense * norm_dense, dim=1, keepdim=True)
    v = torch.cat([norm_dense, d], dim=1)
    Q_dense = torch.bmm(v.unsqueeze(2), v.unsqueeze(1))
    
    sample_count = min(4096, pos_dense.shape[0])
    sample_idx = torch.randperm(pos_dense.shape[0])[:sample_count]
    pos_s, norm_s, Q_s = pos_dense[sample_idx], norm_dense[sample_idx], Q_dense[sample_idx]
    
    dist_s = torch.cdist(pos_s, pos_s)
    _, knn_idx = torch.topk(dist_s, k=31, largest=False)
    edges = torch.stack([torch.arange(len(sample_idx), device=device).view(-1,1).repeat(1,30).flatten(), knn_idx[:,1:].flatten()], dim=0)

    # ==========================================
    # STABLE TEST-TIME OPTIMIZATION
    # ==========================================
    print("Fine-tuning generalized weights (Stable Over-Provisioned TTO)...")
    
    # We re-initialize the assigner to randomly scatter the 800 proxies before TTO
    assigner = SoftAssignmentNetwork(hidden_channels=64, num_proxies=NUM_PROXIES).to(device)
    if os.path.exists(MODEL_PATH):
        checkpoint = torch.load(MODEL_PATH, map_location=device)
        gnn.load_state_dict(checkpoint['gnn'])
        # We intentionally DO NOT load the assigner weights so it uses all 800 fresh proxies
        print(f"Loaded GNN prior from {MODEL_PATH}")

    gnn.train(); assigner.train()
    tto_optimizer = optim.Adam(list(gnn.parameters()) + list(assigner.parameters()), lr=0.002)
    features_s = torch.cat([pos_s, norm_s], dim=1)
    safe_tau = 0.5 
    
    for i in range(300): 
        tto_optimizer.zero_grad()
        S_sampled = assigner(gnn(features_s, edges), tau=safe_tau)
        loss = differentiable_qem_loss(pos_s, Q_s, S_sampled)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(gnn.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(assigner.parameters(), max_norm=1.0)
        
        tto_optimizer.step()
        if (i+1) % 50 == 0: print(f"  TTO Step {i+1} | Loss: {loss.item():.4f}")

    # ==========================================
    # FINAL EXTRACTION
    # ==========================================
    gnn.eval(); assigner.eval()
    with torch.no_grad():
        print("Extracting final dense mesh...")
        S_sampled = assigner(gnn(features_s, edges), tau=safe_tau)
        proxies = torch.matmul(S_sampled.T, pos_s) / (S_sampled.sum(dim=0).unsqueeze(1) + 1e-8)
        
        dist_to_proxies = torch.cdist(pos_dense, proxies)
        vertex_to_proxy = torch.argmin(dist_to_proxies, dim=1)
        
        sim_f = vertex_to_proxy[orig_faces]
        mask = (sim_f[:,0] != sim_f[:,1]) & (sim_f[:,1] != sim_f[:,2]) & (sim_f[:,0] != sim_f[:,2])
        final_f = torch.unique(torch.sort(sim_f[mask], dim=1)[0], dim=0)

    output_filename = f"dense_hybrid_output_{NUM_PROXIES}.obj"
    with open(output_filename, "w") as f:
        for v in proxies.cpu().numpy(): f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for face in (final_f.cpu().numpy() + 1): f.write(f"f {face[0]} {face[1]} {face[2]}\n")
    print(f"Success! {final_f.shape[0]} faces extracted and saved to {output_filename}")

# Run Inference
bunny_path = "dataset/bunny.ply"
if os.path.exists(bunny_path):
    extract_hybrid_mesh(bunny_path)

Running Stable Over-Provisioned Inference on: mps

--- Hybrid Inference Phase: dataset/bunny.ply ---
Fine-tuning generalized weights (Stable Over-Provisioned TTO)...
Loaded GNN prior from universal_vsa_brain.pt
  TTO Step 50 | Loss: 1.6396
  TTO Step 100 | Loss: 1.1828
  TTO Step 150 | Loss: 1.0832
  TTO Step 200 | Loss: 1.0099
  TTO Step 250 | Loss: 0.9828
  TTO Step 300 | Loss: 0.9528
Extracting final dense mesh...
Success! 213 faces extracted and saved to dense_hybrid_output_800.obj


***Unsupervised Inference***

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
import os
import trimesh
import time # Added to track your inference speed!

# --- 1. SETTINGS & HARDWARE ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MODEL_PATH = "universal_vsa_brain.pt"
NUM_PROXIES = 800 # MUST match the number used during your global 155-model training!
print(f"Running ZERO-SHOT Neural VSA on: {device}")

# --- 2. ARCHITECTURE ---
class NeuralVSALayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(NeuralVSALayer, self).__init__(aggr='add') 
        self.mlp = nn.Sequential(nn.Linear(in_channels * 2, 64), nn.ReLU(), nn.Linear(64, out_channels))
    def forward(self, x, edge_index): return self.propagate(edge_index, x=x)
    def message(self, x_i, x_j): return self.mlp(torch.cat([x_i, x_j], dim=1))

class SoftAssignmentNetwork(nn.Module):
    def __init__(self, hidden_channels, num_proxies):
        super().__init__()
        self.predictor = nn.Sequential(nn.Linear(hidden_channels, 64), nn.ReLU(), nn.Linear(64, num_proxies))
    def forward(self, x, tau=1.0): 
        return F.softmax(self.predictor(x) / tau, dim=1)

gnn = NeuralVSALayer(in_channels=6, out_channels=64).to(device)
assigner = SoftAssignmentNetwork(hidden_channels=64, num_proxies=NUM_PROXIES).to(device)

# Load BOTH the GNN and the Assigner (No retraining allowed!)
if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    gnn.load_state_dict(checkpoint['gnn'])
    assigner.load_state_dict(checkpoint['assigner'])
    print(f"Loaded full Universal Brain from {MODEL_PATH}")
else:
    raise FileNotFoundError("Pre-trained weights not found!")

# --- 3. ZERO-SHOT INFERENCE ---
def extract_zero_shot_mesh(input_mesh_path):
    print(f"\n--- Zero-Shot Inference: {input_mesh_path} ---")
    mesh = trimesh.load(input_mesh_path, process=False)
    
    pos_dense = torch.tensor(mesh.vertices, dtype=torch.float32).to(device)
    orig_faces = torch.tensor(mesh.faces, dtype=torch.long).to(device)
    
    pos_dense = pos_dense - pos_dense.mean(dim=0)
    pos_dense = pos_dense / torch.max(torch.sqrt(torch.sum(pos_dense**2, dim=1)))
    
    v0, v1, v2 = pos_dense[orig_faces[:,0]], pos_dense[orig_faces[:,1]], pos_dense[orig_faces[:,2]]
    face_n = torch.cross(v1 - v0, v2 - v0, dim=1)
    vert_n_dense = torch.zeros_like(pos_dense)
    for i in range(3): vert_n_dense.scatter_add_(0, orig_faces[:,i:i+1].expand(-1,3), face_n)
    norm_dense = F.normalize(vert_n_dense, p=2, dim=1)
    
    # Start the timer!
    start_time = time.time()
    
    sample_count = min(4096, pos_dense.shape[0])
    sample_idx = torch.randperm(pos_dense.shape[0])[:sample_count]
    pos_s, norm_s = pos_dense[sample_idx], norm_dense[sample_idx]
    
    dist_s = torch.cdist(pos_s, pos_s)
    _, knn_idx = torch.topk(dist_s, k=31, largest=False)
    edges = torch.stack([torch.arange(len(sample_idx), device=device).view(-1,1).repeat(1,30).flatten(), knn_idx[:,1:].flatten()], dim=0)
    features_s = torch.cat([pos_s, norm_s], dim=1)

    # ==========================================
    # ONE SINGLE FORWARD PASS (NO TRAINING LOOP)
    # ==========================================
    gnn.eval(); assigner.eval()
    with torch.no_grad():
        print("Calculating Neural Assignments...")
        S_sampled = assigner(gnn(features_s, edges), tau=0.5)
        
        # Calculate Proxy Centers
        proxies = torch.matmul(S_sampled.T, pos_s) / (S_sampled.sum(dim=0).unsqueeze(1) + 1e-8)
        
        # Snap vertices to proxies
        dist_to_proxies = torch.cdist(pos_dense, proxies)
        vertex_to_proxy = torch.argmin(dist_to_proxies, dim=1)
        
        # Rebuild faces
        sim_f = vertex_to_proxy[orig_faces]
        mask = (sim_f[:,0] != sim_f[:,1]) & (sim_f[:,1] != sim_f[:,2]) & (sim_f[:,0] != sim_f[:,2])
        final_f = torch.unique(torch.sort(sim_f[mask], dim=1)[0], dim=0)

    inference_time = time.time() - start_time
    print(f"Zero-Shot Inference Time: {inference_time:.4f} seconds")

    output_filename = f"zero_shot_bunny_{len(final_f)}.obj"
    with open(output_filename, "w") as f:
        for v in proxies.cpu().numpy(): f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for face in (final_f.cpu().numpy() + 1): f.write(f"f {face[0]} {face[1]} {face[2]}\n")
    print(f"Success! {final_f.shape[0]} faces extracted and saved to {output_filename}")

# Run Inference
bunny_path = "dataset/bunny.ply"
if os.path.exists(bunny_path):
    extract_zero_shot_mesh(bunny_path)

Running ZERO-SHOT Neural VSA on: mps
Loaded full Universal Brain from universal_vsa_brain.pt

--- Zero-Shot Inference: dataset/bunny.ply ---
Calculating Neural Assignments...
Zero-Shot Inference Time: 0.1237 seconds
Success! 71 faces extracted and saved to zero_shot_bunny_71.obj
